In [1]:
from pathlib import Path
from typing import List, Union

from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
from qwen_agent.agents import FnCallAgent
from qwen_agent.llm.schema import ASSISTANT, Message, USER
from qwen_agent.tools import BaseTool

In [4]:
# 工具一：记录会议主题
class SetTopicTool(BaseTool):
    name = 'set_meeting_topic'
    description = '设置会议主题'
    parameters = {
        'type': 'object',
        'properties': {
            'topic': {'type': 'string', 'description': '会议主题'},
        },
        'required': ['topic'],
    }

    def call(self, params: Union[str, dict], **kwargs) -> str:
        p = self._verify_json_format_args(params)
        topic = p.get('topic', '')
        return f'会议主题已设置为：{topic}'

In [5]:
# 工具二：设置会议时间
class SetTimeTool(BaseTool):
    name = 'set_meeting_time'
    description = '设置会议时间'
    parameters = {
        'type': 'object',
        'properties': {
            'time': {'type': 'string', 'description': '会议时间'},
        },
        'required': ['time'],
    }

    def call(self, params: Union[str, dict], **kwargs) -> str:
        p = self._verify_json_format_args(params)
        time = p.get('time', '')
        return f'会议时间已设置为：{time}'

In [6]:
# 工具三：设置与会人
class SetAttendeesTool(BaseTool):
    name = 'set_meeting_attendees'
    description = '设置与会人员'
    parameters = {
        'type': 'object',
        'properties': {
            'attendees': {'type': 'string', 'description': '人员名单'},
        },
        'required': ['attendees'],
    }

    def call(self, params: Union[str, dict], **kwargs) -> str:
        p = self._verify_json_format_args(params)
        attendees = p.get('attendees', '')
        return f'与会人员已设定为：{attendees}'

In [7]:
def _last_assistant_text(turn_messages: List[Message]) -> str:
    for msg in reversed(turn_messages):
        if msg.role == ASSISTANT and isinstance(msg.content, str) and msg.content.strip():
            return msg.content
    return ''

In [8]:
def chat(agent: FnCallAgent, history: List[Message], text: str) -> str:
    """多轮对话：维护 history，返回本轮助手最终可见回复文本。"""
    history.append(Message(role=USER, content=text))
    last_chunk: List[Message] = []
    for chunk in agent.run(history):
        last_chunk = chunk
    history.extend(last_chunk)
    return _last_assistant_text(last_chunk) or '(无文本回复，可能仅为工具调用)'

In [9]:
# 构建 Agent（函数调用能力用 FnCallAgent，而非抽象基类 Agent）
def build_state_agent():
    llm_cfg = {
        'model': 'qwen-plus',
        'model_server': 'dashscope',
        'generate_cfg': {
            'temperature': 0.3,
            'max_tokens': 512,
        },
    }

    tools = [SetTopicTool(), SetTimeTool(), SetAttendeesTool()]

    agent = FnCallAgent(
        function_list=tools,
        llm=llm_cfg,
        name='MeetingPlannerAgent',
        system_message=(
            '你是一个会议助手Agent，请协助用户完成会议主题、时间和与会人员设定，'
            '需记录所有输入信息以备后续回顾'
        ),
    )

    return agent

In [10]:
agent = build_state_agent()
history: List[Message] = []

2026-04-06 19:19:10,103 - base.py - 95 - INFO - Setting `use_raw_api` to True when using `Qwen3-Max`


In [11]:
print(chat(agent, history, '我要安排一个关于AI发展的会议'))

2026-04-06 19:19:10,131 - base.py - 780 - INFO - ALL tokens: 7, Available tokens: 57973
2026-04-06 19:19:11,182 - base.py - 780 - INFO - ALL tokens: 35, Available tokens: 57973


好的，会议主题已设置为“AI发展”。接下来，请告诉我会议的时间和与会人员名单，以便我帮您完成全部设置。


In [12]:
print(chat(agent, history, '会议时间定在5月3号上午10点'))

2026-04-06 19:19:12,206 - base.py - 780 - INFO - ALL tokens: 77, Available tokens: 57973
2026-04-06 19:19:13,122 - base.py - 780 - INFO - ALL tokens: 117, Available tokens: 57973


好的，会议时间已设置为“5月3号上午10点”。接下来，请提供与会人员名单，我将为您完成最后一步设置。


In [13]:
print(chat(agent, history, '参加人包括张三、李四和王五'))

2026-04-06 19:19:14,170 - base.py - 780 - INFO - ALL tokens: 161, Available tokens: 57973
2026-04-06 19:19:15,160 - base.py - 780 - INFO - ALL tokens: 205, Available tokens: 57973


已为您完成会议全部设置：

- 会议主题：AI发展  
- 会议时间：5月3号上午10点  
- 与会人员：张三、李四、王五  

如需生成会议议程、发送通知或导出日程，欢迎随时告诉我！


In [14]:
print(chat(agent, history, '请总结一下目前的会议信息'))

2026-04-06 19:19:17,049 - base.py - 780 - INFO - ALL tokens: 275, Available tokens: 57973


当前会议信息总结如下：

- **会议主题**：AI发展  
- **会议时间**：5月3号上午10点  
- **与会人员**：张三、李四、王五  

所有基础信息已完整记录，可随时用于后续安排（如发送邀请、准备材料或同步日历）。需要我协助下一步吗？
